In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from psiop_claude import PseudoDifferentialOperator

# Setup plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle("PseudoDifferentialOperator (psiop) — 1D & 2D Examples with Quality Metrics", fontsize=16, fontweight='bold')

# Quality metric helper functions
def rel_l2_error(approx, exact):
    return np.linalg.norm(approx - exact) / np.linalg.norm(exact)

def max_error(approx, exact):
    return np.max(np.abs(approx - exact))

# ==============================================================================
# 1D EXAMPLES
# ==============================================================================

# Setup 1D grid
N1 = 256
x_grid1 = np.linspace(-10, 10, N1, endpoint=False)
dx1 = x_grid1[1] - x_grid1[0]
kx1 = 2 * np.pi * np.fft.fftfreq(N1, d=dx1)

x, xi = sp.symbols('x xi', real=True)
u1 = np.exp(-x_grid1**2)

# ------------------------------------------------------------------------------
# Example 1 (1D): Fractional Laplacian (-d^2/dx^2)^(0.5)
# ------------------------------------------------------------------------------
op_frac_1d = PseudoDifferentialOperator(sp.Abs(xi), [x], mode='symbol')
v1 = op_frac_1d.apply(u1, x_grid1, kx1, freq_window=None)

# Quality assessment: Energy conservation check on real vs imaginary residual
v1_imag_norm = np.linalg.norm(v1.imag) / np.linalg.norm(v1.real)

ax = axes[0, 0]
ax.plot(x_grid1, u1.real, 'b-', label='Input $u(x) = e^{-x^2}$')
ax.plot(x_grid1, v1.real, 'r--', label=r'Applied $(-\Delta)^{1/2} u(x)$')
ax.set_title("1D Ex 1: Fractional Laplacian $|\\xi|$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend()
ax.set_xlim(-5, 5)
ax.text(0.03, 0.05, f"Imaginary Leakage: {v1_imag_norm:.2e}", transform=ax.transAxes, 
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# Example 2 (1D): Variable-Coefficient Operator vs Analytical Solution
# Exact: d/dx(e^-x^2) = -2x e^-x^2, d^2/dx^2(e^-x^2) = (4x^2 - 2) e^-x^2
# ------------------------------------------------------------------------------
expr_1d_var = (1 + 0.5 * sp.sin(x)) * xi**2 + sp.I * sp.cos(x) * xi
op_var_1d = PseudoDifferentialOperator(expr_1d_var, [x], mode='symbol')
v2 = op_var_1d.apply(u1, x_grid1, kx1, freq_window=None)

# Exact analytical result
d1_exact = -2 * x_grid1 * u1
d2_exact = (4 * x_grid1**2 - 2) * u1
v2_exact = (1 + 0.5 * np.sin(x_grid1)) * (-d2_exact) + np.cos(x_grid1) * d1_exact

l2_err_ex2 = rel_l2_error(v2.real, v2_exact)
max_err_ex2 = max_error(v2.real, v2_exact)

ax = axes[1, 0]
ax.plot(x_grid1, u1.real, 'b-', label='Input $u(x)$')
ax.plot(x_grid1, v2.real, 'g-', label=r'Numerical Re$[P(x, D)u]$')
ax.plot(x_grid1, v2_exact, 'r:', label=r'Exact Analytical')
ax.set_title(r"1D Ex 2: Variable Coeff. $(1 + 0.5\sin x)\xi^2 + i\cos(x)\xi$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend()
ax.set_xlim(-5, 5)
ax.text(0.03, 0.05, f"Rel $L_2$ Error: {l2_err_ex2:.2e}\nMax Abs Err: {max_err_ex2:.2e}", 
        transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# Example 3 (1D): Subdomain Boundary Integration
# Fixed LaTeX \le -> \leq
# ------------------------------------------------------------------------------
op_sub_1d = PseudoDifferentialOperator(1 / (1 + xi**2), [x], mode='symbol', apply_backend='direct')

g1 = np.abs(x_grid1) - 2.0      # Level set g <= 0 defines subdomain [-2, 2]
f1 = np.cos(2 * x_grid1)         # Boundary trace
u3 = np.sin(x_grid1)

out_sub1 = op_sub_1d.apply_subdomain(u3, x_grid1, kx1, g1, f1, correction='gain', max_iter=8, assume_local=True)
v_sub = out_sub1['v_Omega'].real

# Residual quality metric outside subdomain (should decay towards 0 outside [-2, 2])
outside_mask = np.abs(x_grid1) > 2.2
outside_residual = np.linalg.norm(v_sub[outside_mask]) / np.linalg.norm(v_sub)

ax = axes[2, 0]
ax.plot(x_grid1, u3, 'k:', label='Original u(x)')
ax.plot(x_grid1, out_sub1['chi_Omega'], 'c--', label=r'Subdomain Indicator $\chi_{\Omega}$')
ax.plot(x_grid1, v_sub, 'r-', label=r'Restricted Result $v_{\Omega}$')
ax.set_title(r"1D Ex 3: Subdomain Integration on $\Omega = \{|x| \leq 2\}$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend()
ax.set_xlim(-6, 6)
ax.text(0.03, 0.05, f"Exterior Leakage Ratio: {outside_residual:.2e}", transform=ax.transAxes, 
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))


# ==============================================================================
# 2D EXAMPLES
# ==============================================================================

# Setup 2D grid
N2 = 128
x_grid2 = np.linspace(-5, 5, N2, endpoint=False)
y_grid2 = np.linspace(-5, 5, N2, endpoint=False)
dx2, dy2 = x_grid2[1] - x_grid2[0], y_grid2[1] - y_grid2[0]

kx2 = 2 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2 * np.pi * np.fft.fftfreq(N2, d=dy2)

X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2d = np.exp(-(X2**2 + Y2**2))

# ------------------------------------------------------------------------------
# Example 4 (2D): 2D Laplacian vs Analytical Solution
# Exact: -\Delta(e^{-(x^2+y^2)}) = 4(1 - x^2 - y^2)e^{-(x^2+y^2)}
# ------------------------------------------------------------------------------
y, eta = sp.symbols('y eta', real=True)
op_lap_2d = PseudoDifferentialOperator(xi**2 + eta**2, [x, y], mode='symbol')

v4 = op_lap_2d.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2, freq_window=None)

v4_exact = 4 * (1 - (X2**2 + Y2**2)) * u2d
l2_err_ex4 = rel_l2_error(v4.real, v4_exact)
max_err_ex4 = max_error(v4.real, v4_exact)

ax = axes[0, 1]
im4 = ax.imshow(v4.real.T, extent=[-5, 5, -5, 5], origin='lower', cmap='magma')
fig.colorbar(im4, ax=ax)
ax.set_title(r"2D Ex 4: 2D Laplacian $(-\Delta u)$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Rel $L_2$ Err: {l2_err_ex4:.2e}\nMax Abs Err: {max_err_ex4:.2e}", 
        transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# Example 5 (2D): Variable Anisotropic Media vs Analytical Derivative
# ------------------------------------------------------------------------------
expr_2d_var = (1 + 0.5 * sp.cos(x)) * xi**2 + (1 + 0.5 * sp.sin(y)) * eta**2
op_var_2d = PseudoDifferentialOperator(expr_2d_var, [x, y], mode='symbol')

v5 = op_var_2d.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2, freq_window=None)

d2x_exact = (4 * X2**2 - 2) * u2d
d2y_exact = (4 * Y2**2 - 2) * u2d
v5_exact = (1 + 0.5 * np.cos(X2)) * (-d2x_exact) + (1 + 0.5 * np.sin(Y2)) * (-d2y_exact)

l2_err_ex5 = rel_l2_error(v5.real, v5_exact)
max_err_ex5 = max_error(v5.real, v5_exact)

ax = axes[1, 1]
im5 = ax.imshow(v5.real.T, extent=[-5, 5, -5, 5], origin='lower', cmap='viridis')
fig.colorbar(im5, ax=ax)
ax.set_title("2D Ex 5: Variable Anisotropic Operator")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Rel $L_2$ Err: {l2_err_ex5:.2e}\nMax Abs Err: {max_err_ex5:.2e}", 
        transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# Example 6 (2D): Directional Derivative vs Analytical Directional Gradient
# Exact: d/d\theta(e^{-(x^2+y^2)}) = -2(x\cos\theta + y\sin\theta)e^{-(x^2+y^2)}
# ------------------------------------------------------------------------------
theta = np.pi / 4
expr_dir = sp.I * (xi * np.cos(theta) + eta * np.sin(theta))
op_dir_2d = PseudoDifferentialOperator(expr_dir, [x, y], mode='symbol')

v6 = op_dir_2d.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2, freq_window=None)

v6_exact = -2 * (X2 * np.cos(theta) + Y2 * np.sin(theta)) * u2d
l2_err_ex6 = rel_l2_error(v6.real, v6_exact)
max_err_ex6 = max_error(v6.real, v6_exact)

ax = axes[2, 1]
im6 = ax.imshow(v6.real.T, extent=[-5, 5, -5, 5], origin='lower', cmap='coolwarm')
fig.colorbar(im6, ax=ax)
ax.set_title(r"2D Ex 6: Directional Derivative ($\theta = \pi/4$)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Rel $L_2$ Err: {l2_err_ex6:.2e}\nMax Abs Err: {max_err_ex6:.2e}", 
        transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Figure setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle("Psiop Subdomain Integration (apply_subdomain) — 1D & 2D Quality Benchmarks", fontsize=15, fontweight='bold')

def rel_l2_error(approx, exact, mask):
    return np.linalg.norm((approx - exact)[mask]) / np.linalg.norm(exact[mask])

# Set up symbolic variables
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# ==============================================================================
# 1D SUBDOMAIN EXAMPLES
# ==============================================================================
N1 = 256
x_grid1 = np.linspace(-10, 10, N1, endpoint=False)
dx1 = x_grid1[1] - x_grid1[0]
kx1 = 2 * np.pi * np.fft.fftfreq(N1, d=dx1)

u1 = np.sin(x_grid1)

# ------------------------------------------------------------------------------
# 1D Ex 1: Helmholtz Green's Filter 1/(1 + \xi^2) on Interval [-3, 3]
# ------------------------------------------------------------------------------
op_1d_1 = PseudoDifferentialOperator(1 / (1 + xi**2), [x], mode='symbol', apply_backend='direct')
g1_1 = np.abs(x_grid1) - 3.0  # Domain: |x| <= 3
f1_1 = np.cos(x_grid1)        # Trace data
mask1_1 = g1_1 <= 0

out1_1 = op_1d_1.apply_subdomain(u1, x_grid1, kx1, g1_1, f1_1, correction='gain', max_iter=8, assume_local=True)
v1_1 = out1_1['v_Omega'].real

# Analytical action: 1/(1 + D^2) sin(x) = 0.5 * sin(x)
v1_1_exact = 0.5 * np.sin(x_grid1)
err1_1 = np.abs(v1_1 - v1_1_exact) * mask1_1
l2_1_1 = rel_l2_error(v1_1, v1_1_exact, mask1_1)

ax = axes[0, 0]
ax.plot(x_grid1, v1_1, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_1_exact, 'k--', label='Exact Solution')
ax.plot(x_grid1, err1_1, 'm:', label='Pointwise Error (in $\Omega$)')
ax.axvspan(-3, 3, color='gray', alpha=0.15, label=r'Subdomain $\Omega$')
ax.set_title("1D Ex 1: Helmholtz Filter on Interval $[-3, 3]$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-6, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_1:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 1D Ex 2: Fractional Advection |\xi|^0.5 on Asymmetric Interval [-1, 4]
# ------------------------------------------------------------------------------
op_1d_2 = PseudoDifferentialOperator(sp.Abs(xi)**0.5, [x], mode='symbol', apply_backend='direct')
g1_2 = (x_grid1 - 1.5)**2 - 6.25  # Domain: [-1, 4]
f1_2 = np.exp(-x_grid1**2)
mask1_2 = g1_2 <= 0

out1_2 = op_1d_2.apply_subdomain(u1, x_grid1, kx1, g1_2, f1_2, correction='gain', max_iter=8, assume_local=True)
v1_2 = out1_2['v_Omega'].real

# Full-space operator reference for comparison
v1_2_ref = op_1d_2.apply(u1, x_grid1, kx1, freq_window=None).real
err1_2 = np.abs(v1_2 - v1_2_ref) * mask1_2
l2_1_2 = rel_l2_error(v1_2, v1_2_ref, mask1_2)

ax = axes[1, 0]
ax.plot(x_grid1, v1_2, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_2_ref, 'k--', label='Full-Space Reference')
ax.plot(x_grid1, err1_2, 'm:', label='Pointwise Error (in $\Omega$)')
ax.axvspan(-1, 4, color='gray', alpha=0.15, label=r'Subdomain $\Omega$')
ax.set_title(r"1D Ex 2: Fractional $|\xi|^{0.5}$ on Asymmetric $[-1, 4]$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-5, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_2:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 1D Ex 3: Variable-Coefficient Diffusion on Disjoint Subdomains
# ------------------------------------------------------------------------------
expr_var_1d = (1 + 0.2 * sp.cos(x)) * xi**2
op_1d_3 = PseudoDifferentialOperator(expr_var_1d, [x], mode='symbol', apply_backend='direct')
g1_3 = np.cos(np.pi * x_grid1 / 3.0)  # Multiple subdomains where cos(...) <= 0
f1_3 = np.zeros_like(x_grid1)
mask1_3 = g1_3 <= 0

u1_gauss = np.exp(-0.5 * x_grid1**2)
out1_3 = op_1d_3.apply_subdomain(u1_gauss, x_grid1, kx1, g1_3, f1_3, correction='gain', max_iter=8, assume_local=True)
v1_3 = out1_3['v_Omega'].real

# Analytical full-space reference: -(1 + 0.2 cos x) * d^2/dx^2(e^{-x^2/2})
d2u = (x_grid1**2 - 1) * u1_gauss
v1_3_exact = (1 + 0.2 * np.cos(x_grid1)) * (-d2u)
err1_3 = np.abs(v1_3 - v1_3_exact) * mask1_3
l2_1_3 = rel_l2_error(v1_3, v1_3_exact, mask1_3)

ax = axes[2, 0]
ax.plot(x_grid1, v1_3, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_3_exact, 'k--', label='Exact Analytical')
ax.plot(x_grid1, err1_3, 'm:', label='Pointwise Error (in $\Omega$)')
ax.plot(x_grid1, mask1_3 * 0.5, 'c--', label='Mask $\chi_{\Omega}$')
ax.set_title("1D Ex 3: Variable Coeff. Operator on Disjoint Domains")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-6, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_3:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))


# ==============================================================================
# 2D SUBDOMAIN EXAMPLES
# ==============================================================================
N2 = 128
x_grid2 = np.linspace(-4, 4, N2, endpoint=False)
y_grid2 = np.linspace(-4, 4, N2, endpoint=False)
dx2, dy2 = x_grid2[1] - x_grid2[0], y_grid2[1] - y_grid2[0]

kx2 = 2 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2 * np.pi * np.fft.fftfreq(N2, d=dy2)

X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2d = np.exp(-(X2**2 + Y2**2))

# ------------------------------------------------------------------------------
# 2D Ex 4: Circular Disk Subdomain for 2D Laplacian
# ------------------------------------------------------------------------------
op_2d_1 = PseudoDifferentialOperator(xi**2 + eta**2, [x, y], mode='symbol', apply_backend='direct')
g2_1 = X2**2 + Y2**2 - 4.0  # Circle R = 2
f2_1 = np.zeros_like(X2)
mask2_1 = g2_1 <= 0

out2_1 = op_2d_1.apply_subdomain(u2d, x_grid2, kx2, g2_1, f2_1, y_grid=y_grid2, ky=ky2, correction='gain', max_iter=8)
v2_1 = out2_1['v_Omega'].real

# Analytical exact result: -\Delta e^{-(x^2+y^2)} = 4(1 - x^2 - y^2)e^{-(x^2+y^2)}
v2_1_exact = 4 * (1 - (X2**2 + Y2**2)) * u2d
err2_1 = np.abs(v2_1 - v2_1_exact) * mask2_1
l2_2_1 = rel_l2_error(v2_1, v2_1_exact, mask2_1)

ax = axes[0, 1]
im1 = ax.imshow(err2_1.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, g2_1, levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im1, ax=ax, label='Absolute Error')
ax.set_title(r"2D Ex 4: Laplacian Error on Disk Domain ($R=2$)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Disk Rel $L_2$ Err: {l2_2_1:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 2D Ex 5: Elliptical Domain for Variable-Coefficient Operator
# ------------------------------------------------------------------------------
expr_2d_var = (1 + 0.3 * sp.cos(x)) * xi**2 + eta**2
op_2d_2 = PseudoDifferentialOperator(expr_2d_var, [x, y], mode='symbol', apply_backend='direct')
g2_2 = (X2 / 2.5)**2 + (Y2 / 1.5)**2 - 1.0  # Ellipse
f2_2 = np.zeros_like(X2)
mask2_2 = g2_2 <= 0

out2_2 = op_2d_2.apply_subdomain(u2d, x_grid2, kx2, g2_2, f2_2, y_grid=y_grid2, ky=ky2, correction='gain', max_iter=8)
v2_2 = out2_2['v_Omega'].real

# Analytical exact comparison
d2x = (4 * X2**2 - 2) * u2d
d2y = (4 * Y2**2 - 2) * u2d
v2_2_exact = (1 + 0.3 * np.cos(X2)) * (-d2x) + (-d2y)
err2_2 = np.abs(v2_2 - v2_2_exact) * mask2_2
l2_2_2 = rel_l2_error(v2_2, v2_2_exact, mask2_2)

ax = axes[1, 1]
im2 = ax.imshow(err2_2.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, g2_2, levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im2, ax=ax, label='Absolute Error')
ax.set_title("2D Ex 5: Variable Operator Error on Ellipse")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Ellipse Rel $L_2$ Err: {l2_2_2:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 2D Ex 6: Square Box Subdomain for Directional Derivative
# ------------------------------------------------------------------------------
theta = np.pi / 3
op_2d_3 = PseudoDifferentialOperator(sp.I * (xi * np.cos(theta) + eta * np.sin(theta)), [x, y], mode='symbol', apply_backend='direct')
g2_3 = np.maximum(np.abs(X2) - 2.0, np.abs(Y2) - 2.0)  # Box [-2, 2] x [-2, 2]
f2_3 = np.zeros_like(X2)
mask2_3 = g2_3 <= 0

out2_3 = op_2d_3.apply_subdomain(u2d, x_grid2, kx2, g2_3, f2_3, y_grid=y_grid2, ky=ky2, correction='gain', max_iter=8)
v2_3 = out2_3['v_Omega'].real

# Analytical exact result: \nabla u \cdot \hat{n}
v2_3_exact = -2 * (X2 * np.cos(theta) + Y2 * np.sin(theta)) * u2d
err2_3 = np.abs(v2_3 - v2_3_exact) * mask2_3
l2_2_3 = rel_l2_error(v2_3, v2_3_exact, mask2_3)

ax = axes[2, 1]
im3 = ax.imshow(err2_3.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, g2_3, levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im3, ax=ax, label='Absolute Error')
ax.set_title(r"2D Ex 6: Directional Derivative Error on Box $[-2, 2]^2$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Box Rel $L_2$ Err: {l2_2_3:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()